# Librerias

In [86]:
import sys
import os

# Agrega la carpeta raíz del proyecto al path de Python
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nbformat
import openpyxl



from src.Data_Processor import DataProcessor



# Carga de datos

In [87]:
procesador = DataProcessor().load_data('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv', 
                                       header=0, 
                                       sep=',', 
                                       decimal='.')


In [88]:
procesador.data.head(30)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


Se renombran las variables para una mejor interpretacion


In [89]:
procesador.data.rename(
    columns={
        'customerID': 'ID_Cliente',
        'gender': 'Genero', 
        'SeniorCitizen': 'Adulto_Mayor',
        'Partner': 'Pareja',
        'Dependents': 'Dependientes',
        'tenure': 'Antiguedad(Meses)',
        'PhoneService': 'Servicio_Telefono',
        'MultipleLines': 'Lineas_Multiples',
        'InternetService': 'Servicio_Internet',
        'OnlineSecurity': 'Seguridad_Online',
        'OnlineBackup': 'Copia_Seguridad_Online',
        'DeviceProtection': 'Proteccion_Dispositivos',
        'TechSupport': 'Soporte_Tecnico',
        'StreamingTV': 'Streaming_TV',
        'StreamingMovies': 'Streaming_Peliculas',
        'Contract': 'Contrato',
        'PaperlessBilling': 'Facturacion_Electronica',
        'PaymentMethod': 'Metodo_Pago',
        'MonthlyCharges': 'Cargos_Mensuales',
        'TotalCharges': 'Cargos_Totales',
        'Churn': 'Abandono'
        }, inplace=True)

## Correcion de tipos

### Numericos

Se corrige el tipo de dato de las variables numericas a "float64".

In [90]:
procesador.data.head(2)

,ID_Cliente,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Servicio_Internet,Seguridad_Online,...,Proteccion_Dispositivos,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargos_Mensuales,Cargos_Totales,Abandono
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No


In [91]:
"""
procesador.set_types(
    {
        "Antiguedad(Meses)": "float64", 
        "Cargos_Mensuales": "float64", 
        "Cargos_Totales": "float64"
    }
)
"""


'\nprocesador.set_types(\n    {\n        "Antiguedad(Meses)": "float64", \n        "Cargos_Mensuales": "float64", \n        "Cargos_Totales": "float64"\n    }\n)\n'

Al tratar de trasformar los datos de la variable "Cargos_Totales" se indica que es imposible convertir " " a float, lo que indica que existen datos en blanco catalogados como str, al ser considerados los espacios como caracteres no se toman como datos nulos o no disponibles, por lo que en la verificacion anterior pasan desapercibidos.

Debido a lo anterior se verificara en el dataset si alguna celda mas se encuentra con espacios en blanco

In [92]:
conteo_espacios = procesador.data.apply(lambda col: col.astype(str).str.isspace().sum()) # Aplica la verificación a todo el dataset columna por columna

print("Cantidad de celdas con solo espacios en blanco por columna:")
print(conteo_espacios[conteo_espacios > 0]) # Muestra solo las columnas que tengan problemas con ' '


Cantidad de celdas con solo espacios en blanco por columna:
Cargos_Totales    11
dtype: int64


Gracias a la busqueda anterir se encuentran 11 filas en la variable "Cargos_Totales" por lo que se estudiara mas a fondo el porque de los espacios en blanco.

In [93]:
procesador.data[procesador.data['Cargos_Totales'] == ' ']

,ID_Cliente,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Servicio_Internet,Seguridad_Online,...,Proteccion_Dispositivos,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargos_Mensuales,Cargos_Totales,Abandono
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


In [94]:
procesador.data[procesador.data['Antiguedad(Meses)'] == 0]



,ID_Cliente,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Servicio_Internet,Seguridad_Online,...,Proteccion_Dispositivos,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargos_Mensuales,Cargos_Totales,Abandono
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


In [95]:
procesador.data[procesador.data['Antiguedad(Meses)'] == 1]

,ID_Cliente,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Servicio_Internet,Seguridad_Online,...,Proteccion_Dispositivos,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargos_Mensuales,Cargos_Totales,Abandono
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
20,8779-QRDMV,Male,1,No,No,1,No,No phone service,DSL,No,...,Yes,No,No,Yes,Month-to-month,Yes,Electronic check,39.65,39.65,Yes
22,1066-JKSGK,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes
27,8665-UTDHZ,Male,0,Yes,Yes,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,30.20,30.2,Yes
33,7310-EGVHZ,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),20.20,20.2,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6979,5351-QESIO,Male,0,No,Yes,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,No,Mailed check,24.20,24.2,No
7010,0723-DRCLG,Female,1,Yes,No,1,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,74.45,74.45,Yes
7016,1471-GIQKQ,Female,0,No,No,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,49.95,49.95,No
7018,1122-JWTJW,Male,0,Yes,Yes,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,70.65,70.65,Yes


In [96]:
procesador.data[procesador.data['Antiguedad(Meses)'] == 0].shape == procesador.data[procesador.data['Cargos_Totales'] == ' '].shape

True

Al profundizar mas en las 11 variables con datos en blanco se encuentra que los individuos afectados cuentan con valores iguales a 0 en la variable "Antiguedad(Meses)", apartir de una antiguedad mayor a 1 se cuenta con un valor en "Cargos_Totales" que indica el monto total que se adquirio a lo largo de los periodos establecidos segun "Cargos_Mensuales".

Debido a esto se puede inferir que los datos no representan cobros por parte de la organizacion al abonado ya que este dimitio del servicio antes de terminar el primer mes de suscripción.

Al no existir un cobro por parte de la organización, los cargos totales quedan en ceros, valor que reemplazará los valores en blanco.

In [97]:
procesador.data['Cargos_Totales'] = procesador.data['Cargos_Totales'].replace(r'^\s*$', '0', regex=True)

procesador.set_types(
    {
        "Antiguedad(Meses)": "float64", 
        "Cargos_Mensuales": "float64", 
        "Cargos_Totales": "float64"
    }
)

In [98]:
procesador.data[procesador.data['Cargos_Totales'] == ' ']

,ID_Cliente,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Servicio_Internet,Seguridad_Online,...,Proteccion_Dispositivos,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargos_Mensuales,Cargos_Totales,Abandono


In [99]:
procesador.data[procesador.data['Antiguedad(Meses)'] == 0]

,ID_Cliente,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Servicio_Internet,Seguridad_Online,...,Proteccion_Dispositivos,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargos_Mensuales,Cargos_Totales,Abandono
488,4472-LVYGI,Female,0,Yes,Yes,0.0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,0.0,No
753,3115-CZMZD,Male,0,No,Yes,0.0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,0.0,No
936,5709-LVOEQ,Female,0,Yes,Yes,0.0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,0.0,No
1082,4367-NUYAO,Male,0,Yes,Yes,0.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,0.0,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0.0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,0.0,No
3331,7644-OMVMY,Male,0,Yes,Yes,0.0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,0.0,No
3826,3213-VVOLG,Male,0,Yes,Yes,0.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,0.0,No
4380,2520-SGTTA,Female,0,Yes,Yes,0.0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,0.0,No
5218,2923-ARZLG,Male,0,Yes,Yes,0.0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,0.0,No
6670,4075-WKNIU,Female,0,Yes,Yes,0.0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,0.0,No


In [100]:
conteo_espacios = procesador.data.apply(lambda col: col.astype(str).str.isspace().sum()) # Aplica la verificación a todo el dataset columna por columna

print("Cantidad de celdas con solo espacios en blanco por columna:")
print(conteo_espacios[conteo_espacios > 0]) # Muestra solo las columnas que tengan problemas con ' '


Cantidad de celdas con solo espacios en blanco por columna:
Series([], dtype: int64)


In [101]:
procesador.data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID_Cliente               7043 non-null   str    
 1   Genero                   7043 non-null   str    
 2   Adulto_Mayor             7043 non-null   int64  
 3   Pareja                   7043 non-null   str    
 4   Dependientes             7043 non-null   str    
 5   Antiguedad(Meses)        7043 non-null   float64
 6   Servicio_Telefono        7043 non-null   str    
 7   Lineas_Multiples         7043 non-null   str    
 8   Servicio_Internet        7043 non-null   str    
 9   Seguridad_Online         7043 non-null   str    
 10  Copia_Seguridad_Online   7043 non-null   str    
 11  Proteccion_Dispositivos  7043 non-null   str    
 12  Soporte_Tecnico          7043 non-null   str    
 13  Streaming_TV             7043 non-null   str    
 14  Streaming_Peliculas      7043 non-n

Los cambios en las celdas con ' ' se realizaron correctamente por 0 y el cambio de tipos no indico ningun problema.

In [102]:
procesador.data.sample(n=20, axis=0)

,ID_Cliente,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Servicio_Internet,Seguridad_Online,...,Proteccion_Dispositivos,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargos_Mensuales,Cargos_Totales,Abandono
2074,6179-GJPSO,Female,1,No,No,43.0,Yes,No,Fiber optic,Yes,...,No,Yes,Yes,Yes,Month-to-month,Yes,Credit card (automatic),108.15,4600.70,Yes
5121,4903-CNOZC,Male,0,No,No,29.0,Yes,Yes,DSL,No,...,Yes,No,No,Yes,One year,No,Credit card (automatic),70.90,1964.60,No
5779,2237-ZFSMY,Female,0,No,No,39.0,Yes,No,Fiber optic,Yes,...,Yes,Yes,Yes,No,One year,Yes,Electronic check,95.55,3692.85,Yes
2416,8982-NHAVY,Male,0,No,No,27.0,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Bank transfer (automatic),100.50,2673.45,No
5839,2077-MPJQO,Male,1,Yes,No,7.0,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,75.40,533.05,No
1544,3794-NFNCH,Male,0,Yes,No,30.0,Yes,No,DSL,No,...,Yes,No,No,No,Month-to-month,Yes,Mailed check,50.00,1474.90,No
1795,5922-ABDVO,Female,0,Yes,No,19.0,Yes,Yes,Fiber optic,No,...,Yes,No,No,Yes,Month-to-month,No,Mailed check,89.65,1761.05,Yes
5251,8485-GJCDN,Female,1,No,No,5.0,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,69.05,318.50,Yes
6447,8782-NUUOL,Male,0,No,No,60.0,Yes,No,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,No,Mailed check,79.00,4801.10,No
460,4719-UMSIY,Male,0,No,Yes,6.0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.65,100.90,No


### Dicotomicas

Las variables dicotomicas se presentan es formato "1" y "0" como enteros y "Yes" y "No" como caracter, por lo que se deben de tratar para mantener coherencia sobre los tipos.

Los valores entreros se pueden combertir directamente, pero los terminos en inglés se deben de mapear para un correcto cambio de tipos.

In [103]:
columnas = ['Dependientes', 'Pareja', 'Adulto_Mayor', 'Servicio_Telefono', 'Facturacion_Electronica', 'Abandono']

for columna in columnas:
    print(f"Valores en la columna '{columna}':")
    print(procesador.data[columna].value_counts())
    print("-" * 30)

Valores en la columna 'Dependientes':
Dependientes
No     4933
Yes    2110
Name: count, dtype: int64
------------------------------
Valores en la columna 'Pareja':
Pareja
No     3641
Yes    3402
Name: count, dtype: int64
------------------------------
Valores en la columna 'Adulto_Mayor':
Adulto_Mayor
0    5901
1    1142
Name: count, dtype: int64
------------------------------
Valores en la columna 'Servicio_Telefono':
Servicio_Telefono
Yes    6361
No      682
Name: count, dtype: int64
------------------------------
Valores en la columna 'Facturacion_Electronica':
Facturacion_Electronica
Yes    4171
No     2872
Name: count, dtype: int64
------------------------------
Valores en la columna 'Abandono':
Abandono
No     5174
Yes    1869
Name: count, dtype: int64
------------------------------


In [104]:
# Cuenta el total de celdas NaN que tiene cada columna
procesador.data.isna().sum()

# Si quieres ver solo las columnas que tienen AL MENOS un nulo:
conteo_nulos = procesador.data.isna().sum()
print(conteo_nulos[conteo_nulos > 0])


Series([], dtype: int64)


In [105]:
procesador.replace_to_numeric_bool(
    ['Dependientes', 
     'Pareja', 
     'Servicio_Telefono', 
     'Facturacion_Electronica', 
     'Abandono'],
    {'Yes': 1, 'No': 0}
).replace_to_numeric_bool(
    ['Seguridad_Online', 
     'Copia_Seguridad_Online', 
     'Proteccion_Dispositivos', 
     'Soporte_Tecnico', 
     'Streaming_TV', 
     'Streaming_Peliculas'],
    {'Yes': 1, 'No': 0},
    True,
    to_replace = ['No internet service'], 
    value = 'No'
).replace_to_numeric_bool(
    ['Lineas_Multiples'],
    {'Yes': 1, 'No': 0},
    True,
    to_replace = ['No phone service'], 
    value = 'No'
)

In [106]:
for columna in columnas:
    print(f"Valores en la columna '{columna}':")
    print(procesador.data[columna].value_counts())
    print("-" * 30)

Valores en la columna 'Dependientes':
Dependientes
0    4933
1    2110
Name: count, dtype: int64
------------------------------
Valores en la columna 'Pareja':
Pareja
0    3641
1    3402
Name: count, dtype: int64
------------------------------
Valores en la columna 'Adulto_Mayor':
Adulto_Mayor
0    5901
1    1142
Name: count, dtype: int64
------------------------------
Valores en la columna 'Servicio_Telefono':
Servicio_Telefono
1    6361
0     682
Name: count, dtype: int64
------------------------------
Valores en la columna 'Facturacion_Electronica':
Facturacion_Electronica
1    4171
0    2872
Name: count, dtype: int64
------------------------------
Valores en la columna 'Abandono':
Abandono
0    5174
1    1869
Name: count, dtype: int64
------------------------------


In [107]:
procesador.data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID_Cliente               7043 non-null   str    
 1   Genero                   7043 non-null   str    
 2   Adulto_Mayor             7043 non-null   int64  
 3   Pareja                   7043 non-null   int64  
 4   Dependientes             7043 non-null   int64  
 5   Antiguedad(Meses)        7043 non-null   float64
 6   Servicio_Telefono        7043 non-null   int64  
 7   Lineas_Multiples         7043 non-null   int64  
 8   Servicio_Internet        7043 non-null   str    
 9   Seguridad_Online         7043 non-null   int64  
 10  Copia_Seguridad_Online   7043 non-null   int64  
 11  Proteccion_Dispositivos  7043 non-null   int64  
 12  Soporte_Tecnico          7043 non-null   int64  
 13  Streaming_TV             7043 non-null   int64  
 14  Streaming_Peliculas      7043 non-n

In [108]:
procesador.data.nunique()

ID_Cliente                 7043
Genero                        2
Adulto_Mayor                  2
Pareja                        2
Dependientes                  2
Antiguedad(Meses)            73
Servicio_Telefono             2
Lineas_Multiples              2
Servicio_Internet             3
Seguridad_Online              2
Copia_Seguridad_Online        2
Proteccion_Dispositivos       2
Soporte_Tecnico               2
Streaming_TV                  2
Streaming_Peliculas           2
Contrato                      3
Facturacion_Electronica       2
Metodo_Pago                   4
Cargos_Mensuales           1585
Cargos_Totales             6531
Abandono                      2
dtype: int64

Con el cambio de tipos en el dataset quedan 6 variables dicotomicas, 3 numericas y 12 categoricas, esto em parte es correcto pero al final de todo las categoricas solo serian 11 ya que pandas cuenta el ID como variable.

## Trata de variables categoricas

#### Cuenta de valores unicos por variable

In [109]:
columnas_cat = ["Servicio_Internet", 
      "Contrato",
      "Metodo_Pago"]

procesador.data[columnas_cat].nunique()

Servicio_Internet    3
Contrato             3
Metodo_Pago          4
dtype: int64

In [110]:
for variable in columnas_cat:
    print(f"{variable}: \n \t{list(procesador.data[variable].unique())}\n")

Servicio_Internet: 
 	['DSL', 'Fiber optic', 'No']

Contrato: 
 	['Month-to-month', 'One year', 'Two year']

Metodo_Pago: 
 	['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']



Dentro de los valores unicos de las variables categoricas se presenta una variable ordinal que seria 'Contrato', el resto se presentan como variables nominales.

El resto de variables nominales, con exepcion de "Metodo_Pago" y "Servicio_Internet",  presentan redundancias con respecto a la informacion general que contiene el dataset, ya hay columnas booleanas que indican los servicios de los que dispone un abonado, por lo que si en la casilla que indica el servicio se indica que no lo tienen suscrito, por consecuencia las prestaciones a ese servicio no estarian disponibles o asociadas con el abonado, por lo que indicar 'No internet service' o 'No phone service' en el resto de variables solo aporta redundancia en las categorias y solo genera ruido en el analisis.

Por lo mencionado anteriormente se toma la decicion de tratar estas variables como dicotomicas pasando las que indican 'No internet service' o 'No phone service' como 'False' para mantener una mejor interpretacion de los datos, la columna 'Servicio_Internet' se pasara a codigo disyuntivo completo sin tomar en cuenta la categoria 'No', si un cliente tiene 0 en 'DSL' y 0 en 'Fibra Óptica', por consecuente pertenece a la categoría "No" aunque no se indique explicitamente, y "Contrato" se combertira en numerica para indicar tratar de manejar la mayor cantidad de valores posibles en forma numerica, siendo 'Month-to-month' igual a 0, 'One year' igual a 1 y 'Two year' igual 2.

In [111]:
procesador.set_categorical_columns(
    columna = 'Contrato',
    categorias = ['Month-to-month', 'One year', 'Two year'], 
    cat_codes=True,
    ordered=True
).set_categorical_columns(
    columna = 'Genero',
    categorias = ['Male', 'Female'], 
    ordered=False
).get_dummies(
    columna = 'Metodo_Pago',
    prefijo = 'MP',
    categorias = ['Electronic check', 
                'Mailed check', 
                'Bank transfer (automatic)', 
                'Credit card (automatic)'], 
    ordered=False
).get_dummies(
    columna = 'Servicio_Internet',
    prefijo = 'Internet'
)


Index(['MP_Electronic check', 'MP_Mailed check',
       'MP_Bank transfer (automatic)'],
      dtype='str')
Index(['Internet_DSL', 'Internet_Fiber optic'], dtype='str')


In [112]:
procesador.data.nunique()

ID_Cliente                      7043
Genero                             2
Adulto_Mayor                       2
Pareja                             2
Dependientes                       2
Antiguedad(Meses)                 73
Servicio_Telefono                  2
Lineas_Multiples                   2
Internet_DSL                       2
Internet_Fiber optic               2
Seguridad_Online                   2
Copia_Seguridad_Online             2
Proteccion_Dispositivos            2
Soporte_Tecnico                    2
Streaming_TV                       2
Streaming_Peliculas                2
Contrato                           3
Facturacion_Electronica            2
MP_Electronic check                2
MP_Mailed check                    2
MP_Bank transfer (automatic)       2
Cargos_Mensuales                1585
Cargos_Totales                  6531
Abandono                           2
dtype: int64

In [113]:

# Se elimina la columna 'ID_Cliente' ya que no es relevante para el análisis
procesador.data.drop(columns=['ID_Cliente'], inplace=True)  


In [114]:
procesador.data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   Genero                        7043 non-null   category
 1   Adulto_Mayor                  7043 non-null   int64   
 2   Pareja                        7043 non-null   int64   
 3   Dependientes                  7043 non-null   int64   
 4   Antiguedad(Meses)             7043 non-null   float64 
 5   Servicio_Telefono             7043 non-null   int64   
 6   Lineas_Multiples              7043 non-null   int64   
 7   Internet_DSL                  7043 non-null   int8    
 8   Internet_Fiber optic          7043 non-null   int8    
 9   Seguridad_Online              7043 non-null   int64   
 10  Copia_Seguridad_Online        7043 non-null   int64   
 11  Proteccion_Dispositivos       7043 non-null   int64   
 12  Soporte_Tecnico               7043 non-null   int64   
 13 

In [115]:
procesador.data.nunique()

Genero                             2
Adulto_Mayor                       2
Pareja                             2
Dependientes                       2
Antiguedad(Meses)                 73
Servicio_Telefono                  2
Lineas_Multiples                   2
Internet_DSL                       2
Internet_Fiber optic               2
Seguridad_Online                   2
Copia_Seguridad_Online             2
Proteccion_Dispositivos            2
Soporte_Tecnico                    2
Streaming_TV                       2
Streaming_Peliculas                2
Contrato                           3
Facturacion_Electronica            2
MP_Electronic check                2
MP_Mailed check                    2
MP_Bank transfer (automatic)       2
Cargos_Mensuales                1585
Cargos_Totales                  6531
Abandono                           2
dtype: int64

In [116]:
procesador.data.head(50)


,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Internet_DSL,Internet_Fiber optic,Seguridad_Online,...,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,MP_Electronic check,MP_Mailed check,MP_Bank transfer (automatic),Cargos_Mensuales,Cargos_Totales,Abandono
0,Female,0,1,0,1.0,0,0,1,0,0,...,0,0,0,1,1,0,0,29.85,29.85,0
1,Male,0,0,0,34.0,1,0,1,0,1,...,0,0,1,0,0,1,0,56.95,1889.50,0
2,Male,0,0,0,2.0,1,0,1,0,1,...,0,0,0,1,0,1,0,53.85,108.15,1
3,Male,0,0,0,45.0,0,0,1,0,1,...,0,0,1,0,0,0,1,42.30,1840.75,0
4,Female,0,0,0,2.0,1,0,0,1,0,...,0,0,0,1,1,0,0,70.70,151.65,1
5,Female,0,0,0,8.0,1,1,0,1,0,...,1,1,0,1,1,0,0,99.65,820.50,1
6,Male,0,0,1,22.0,1,1,0,1,0,...,1,0,0,1,0,0,0,89.10,1949.40,0
7,Female,0,0,0,10.0,0,0,1,0,1,...,0,0,0,0,0,1,0,29.75,301.90,0
8,Female,0,1,0,28.0,1,1,0,1,0,...,1,1,0,1,1,0,0,104.80,3046.05,1
9,Male,0,0,1,62.0,1,0,1,0,1,...,0,0,1,0,0,0,1,56.15,3487.95,0


## Guardado del dataset

In [117]:
# Mantiene los booleanos y categorías exactamente como se configuraron
procesador.save_data('../data/processed/Proyecto_B_Clientes_Limpio.parquet', index=False)